In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


/kaggle/input/datasets/jvkrishwanth/extracted/df_main_phase1.csv
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_27_MW-ica.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_14_MW-ica-metadata.mat
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_3_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_22_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_15_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_16_MW-ica-metadata.mat
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_23_MW-ica.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_19_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED/P_7_MW-clean-epo.fif
/kaggle/input/datasets/jvkrishwanth/extracted/extracted_datas

In [2]:
from pathlib import Path
import re
import warnings

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, hilbert, sosfiltfilt, welch
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


RAW_DIR = Path("/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset")
CLEAN_DIR = Path("/kaggle/input/datasets/jvkrishwanth/extracted/extracted_dataset_ICA_CLEANED")
LABEL_CSV = Path("/kaggle/input/datasets/jvkrishwanth/extracted/df_main_phase1.csv")
OUT_DIR = Path("/kaggle/working/raw_vs_clean_deepmlp")

CHANNELS = [
    "Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T3", "C3", "Cz", "C4",
    "T4", "T5", "P3", "Pz", "P4", "T6", "O1", "O2",
]
BANDS = {
    "Delta": (1, 4),
    "Theta": (4, 8),
    "Alpha": (8, 12),
    "Beta": (13, 30),
    "Gamma": (30, 45),
}
WINDOW_SECONDS = 2.0
STEP_SECONDS = 0.5
N_WINDOWS = 6
RANDOM_STATE = 42
N_OUTER_SPLITS = 5
N_INNER_SPLITS = 4
MAX_EPOCHS = 100
PATIENCE = 15
BATCH_SIZE = 64
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def canonical_channel(name: str) -> str:
    """Make common FIF spellings match the canonical 19-channel montage."""
    aliases = {"FP1": "Fp1", "FP2": "Fp2", "PZ": "Pz", "CZ": "Cz", "FZ": "Fz"}
    return aliases.get(name.upper(), name)


def bandpass(data, low, high, sfreq):
    sos = butter(4, [low / (sfreq / 2), high / (sfreq / 2)], btype="band", output="sos")
    return sosfiltfilt(sos, data, axis=-1)


def burst_count(envelope):
    threshold = envelope.mean() + 2 * envelope.std()
    return float(np.sum(np.diff((envelope > threshold).astype(int)) == 1))


def subject_id(value) -> int:
    """Handles labels such as 1, 'P_1', 'sub-01', and '01'."""
    found = re.findall(r"\d+", str(value))
    if not found:
        raise ValueError(f"Cannot obtain a numeric subject id from {value!r}")
    return int(found[-1])


def subject_labels(df_main, sub_id):
    sub = df_main[df_main["Subject"].map(subject_id) == sub_id]
    # df_main_phase1 was created from the ICA-cleaned FIF files. Epoch_Index is
    # therefore the original trial index retained by the cleaned file, rather
    # than a consecutive row number. Preserve it as the label lookup key.
    epochs = sub.groupby(["Session", "Epoch_Index"], sort=True)["State"].first()
    if epochs.index.get_level_values("Session").nunique() != 1:
        raise ValueError(f"Subject {sub_id} has multiple sessions; add Session to the FIF mapping")
    labels = (epochs.astype(str).str.upper() == "MW").astype(int)
    labels.index = labels.index.get_level_values("Epoch_Index").astype(int)
    return labels


def load_epochs(fif_path):
    epochs = mne.read_epochs(fif_path, preload=True, verbose=False)
    rename = {name: canonical_channel(name) for name in epochs.ch_names}
    epochs.rename_channels(rename)
    missing = sorted(set(CHANNELS) - set(epochs.ch_names))
    if missing:
        raise ValueError(f"{fif_path.name} lacks required channels: {missing}")
    return epochs.copy().pick(CHANNELS)


def pair_raw_and_clean_epochs(raw_epochs, clean_epochs, labels_by_source_index, sub_id):
    """Keep raw epochs that correspond to epochs surviving ICA cleaning.

    ICA rejection legitimately reduces the number of clean epochs.  Matching on
    the original MNE event sample and event code makes the raw/clean comparison
    paired: both conditions contain precisely the same trials and labels.
    """
    raw_keys = [tuple(event[[0, 2]]) for event in raw_epochs.events]
    clean_keys = [tuple(event[[0, 2]]) for event in clean_epochs.events]
    clean_positions = {key: idx for idx, key in enumerate(clean_keys)}
    raw_idx = [idx for idx, key in enumerate(raw_keys) if key in clean_positions]
    clean_idx = [clean_positions[raw_keys[idx]] for idx in raw_idx]
    if not raw_idx:
        raise ValueError("no matching raw/clean event samples were found")
    clean_source_indices = np.asarray(clean_epochs.selection, dtype=int)[clean_idx]
    missing = sorted(set(clean_source_indices) - set(labels_by_source_index.index))
    if missing:
        raise ValueError(f"labels missing for cleaned source epoch indices: {missing}")
    paired_labels = labels_by_source_index.loc[clean_source_indices].to_numpy(dtype=int)
    print(f"Subject {sub_id}: using {len(raw_idx)} paired epochs "
          f"(raw={len(raw_epochs)}, clean={len(clean_epochs)})")
    return raw_epochs[raw_idx], clean_epochs[clean_idx], paired_labels


def extract_features(epochs, labels, sub_id, condition):
    """Return six overlapping-window feature rows for every labeled epoch."""
    data = epochs.get_data(copy=True)
    if len(data) != len(labels):
        raise ValueError(
            f"Subject {sub_id}: {condition} has {len(data)} FIF epochs but {len(labels)} labels. "
            "This subject is skipped so labels are never misaligned."
        )

    sfreq = float(epochs.info["sfreq"])
    win_len, step = int(WINDOW_SECONDS * sfreq), int(STEP_SECONDS * sfreq)
    if data.shape[-1] < win_len + (N_WINDOWS - 1) * step:
        raise ValueError(f"{condition} epochs are too short for {N_WINDOWS} windows.")

    alpha_env = np.abs(hilbert(bandpass(data, 8, 12, sfreq), axis=-1))
    theta_env = np.abs(hilbert(bandpass(data, 4, 8, sfreq), axis=-1))
    rows = []

    for epoch_idx, label in enumerate(labels):
        for window_idx in range(N_WINDOWS):
            start, end = window_idx * step, window_idx * step + win_len
            raw = data[epoch_idx, :, start:end]
            a_env = alpha_env[epoch_idx, :, start:end]
            t_env = theta_env[epoch_idx, :, start:end]
            freqs, psd = welch(raw, fs=sfreq, nperseg=256, noverlap=128, axis=-1)
            total_mask = (freqs >= 1) & (freqs <= 45)
            alpha_corr = np.nan_to_num(np.corrcoef(a_env))

            row = {
                "Condition": condition,
                "Subject": sub_id,
                "Epoch_Index": epoch_idx,
                "Window_Index": window_idx,
                "Label": label,
            }
            for channel_idx, channel in enumerate(CHANNELS):
                channel_psd = psd[channel_idx]
                total = channel_psd[total_mask].mean()
                for band, (low, high) in BANDS.items():
                    mask = (freqs >= low) & (freqs < high if band != "Gamma" else freqs <= high)
                    absolute = channel_psd[mask].mean()
                    row[f"{channel}_{band}_Abs"] = absolute
                    row[f"{channel}_{band}_Rel"] = absolute / total if total > 0 else 0.0

                for prefix, env in (("Alpha", a_env[channel_idx]), ("Theta", t_env[channel_idx])):
                    mean = env.mean()
                    row[f"{channel}_{prefix}_Mean"] = mean
                    row[f"{channel}_{prefix}_Var"] = env.var()
                    row[f"{channel}_{prefix}_CV"] = env.std() / mean if mean > 0 else 0.0
                    row[f"{channel}_{prefix}_Bursts"] = burst_count(env)
                row[f"{channel}_Envelope_Sync"] = alpha_corr[channel_idx, np.arange(len(CHANNELS)) != channel_idx].mean()
            rows.append(row)
    return rows


def find_fif(folder, sub_id, cleaned):
    suffix = "-clean-epo.fif" if cleaned else "-epo.fif"
    exact = folder / f"P_{sub_id}_MW{suffix}"
    if exact.exists():
        return exact
    # Accommodates alternate capitalization/naming while still selecting the intended subject.
    candidates = [p for p in folder.rglob("*.fif") if re.search(rf"P_?{sub_id}(?!\d)", p.name, re.I)]
    candidates = [p for p in candidates if ("clean" in p.name.lower()) == cleaned and p.name.lower().endswith("-epo.fif")]
    return candidates[0] if len(candidates) == 1 else None


def make_dataset(df_main):
    rows = []
    for sub_id in sorted({subject_id(value) for value in df_main["Subject"].unique()}):
        raw_path = find_fif(RAW_DIR, sub_id, cleaned=False)
        clean_path = find_fif(CLEAN_DIR, sub_id, cleaned=True)
        if raw_path is None or clean_path is None:
            print(f"Skipping subject {sub_id}: missing raw or cleaned FIF.")
            continue
        try:
            labels_by_source_index = subject_labels(df_main, sub_id)
            raw_epochs, clean_epochs = load_epochs(raw_path), load_epochs(clean_path)
            raw_epochs, clean_epochs, paired_labels = pair_raw_and_clean_epochs(
                raw_epochs, clean_epochs, labels_by_source_index, sub_id
            )
            rows.extend(extract_features(raw_epochs, paired_labels, sub_id, "Raw"))
            rows.extend(extract_features(clean_epochs, paired_labels, sub_id, "ICA-cleaned"))
            print(f"Finished subject {sub_id}")
        except ValueError as exc:
            warnings.warn(f"Skipping subject {sub_id}: {exc}")
    return pd.DataFrame(rows)


class DeepMLP(nn.Module):
    """Binary classifier for the window-level EEG feature vectors."""
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.40),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.30),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(1)


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_loader(X, y, shuffle=False):
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32),
                            torch.tensor(y, dtype=torch.float32))
    # BatchNorm requires batches of at least two observations.
    drop_last = shuffle and len(dataset) > BATCH_SIZE and len(dataset) % BATCH_SIZE == 1
    return DataLoader(dataset, batch_size=min(BATCH_SIZE, len(dataset)), shuffle=shuffle,
                      drop_last=drop_last, num_workers=0,
                      pin_memory=torch.cuda.is_available())


def train_deep_mlp(X_train, y_train, X_val, y_val, fold_seed):
    set_seed(fold_seed)
    model = DeepMLP(X_train.shape[1]).to(DEVICE)
    n_positive = max(int(y_train.sum()), 1)
    pos_weight = torch.tensor([(len(y_train) - n_positive) / n_positive], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=5, factor=0.5
    )
    train_loader, val_loader = make_loader(X_train, y_train, True), make_loader(X_val, y_val)
    best_state, best_loss, stale_epochs = None, np.inf, 0

    for _ in range(MAX_EPOCHS):
        model.train()
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(x_batch.to(DEVICE)), y_batch.to(DEVICE))
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss_sum = 0.0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                loss = criterion(model(x_batch.to(DEVICE)), y_batch.to(DEVICE))
                val_loss_sum += loss.item() * len(x_batch)
        val_loss = val_loss_sum / len(val_loader.dataset)
        scheduler.step(val_loss)

        if val_loss < best_loss:
            best_loss, stale_epochs = val_loss, 0
            best_state = {name: value.detach().cpu().clone()
                          for name, value in model.state_dict().items()}
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return model


def predict_probabilities(model, X):
    loader = make_loader(X, np.zeros(len(X)))
    model.eval()
    probabilities = []
    with torch.no_grad():
        for x_batch, _ in loader:
            probabilities.extend(torch.sigmoid(model(x_batch.to(DEVICE))).cpu().numpy())
    return np.asarray(probabilities)


def evaluate_condition(df, condition):
    subset = df[df["Condition"] == condition].reset_index(drop=True)
    metadata = {"Condition", "Subject", "Epoch_Index", "Window_Index", "Label"}
    features = [column for column in subset.columns if column not in metadata]
    X = np.nan_to_num(subset[features].to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    y = subset["Label"].to_numpy(dtype=int)
    # All overlapping windows from an original EEG epoch remain in a single fold.
    groups = (subset["Subject"].astype(str) + "_" + subset["Epoch_Index"].astype(str)).to_numpy()
    outer_cv = StratifiedGroupKFold(n_splits=N_OUTER_SPLITS, shuffle=True,
                                   random_state=RANDOM_STATE)
    predictions, probabilities = np.zeros(len(y), dtype=int), np.zeros(len(y), dtype=float)

    for fold, (train_val_idx, test_idx) in enumerate(outer_cv.split(X, y, groups), start=1):
        # An inner grouped split is reserved only for early stopping. Test-fold
        # labels are never used for scaling, training, or model selection.
        inner_cv = StratifiedGroupKFold(n_splits=N_INNER_SPLITS, shuffle=True,
                                       random_state=RANDOM_STATE + fold)
        train_idx, val_idx = next(inner_cv.split(X[train_val_idx], y[train_val_idx],
                                                groups[train_val_idx]))
        train_idx, val_idx = train_val_idx[train_idx], train_val_idx[val_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X[train_idx])
        X_val = scaler.transform(X[val_idx])
        X_test = scaler.transform(X[test_idx])
        model = train_deep_mlp(X_train, y[train_idx], X_val, y[val_idx],
                               RANDOM_STATE + fold)
        probabilities[test_idx] = predict_probabilities(model, X_test)
        predictions[test_idx] = (probabilities[test_idx] >= 0.5).astype(int)
        print(f"{condition} fold {fold}/{N_OUTER_SPLITS} complete")

    # Average the window probabilities per unique epoch (group)
    unique_groups, first_indices = np.unique(groups, return_index=True)
    epoch_y = y[first_indices]
    epoch_probabilities = np.zeros(len(unique_groups))
    epoch_predictions = np.zeros(len(unique_groups), dtype=int)
    
    for idx, g in enumerate(unique_groups):
        mask = groups == g
        epoch_probabilities[idx] = probabilities[mask].mean()
        epoch_predictions[idx] = int(epoch_probabilities[idx] >= 0.5)

    metrics = {
        "Condition": condition,
        "Accuracy": accuracy_score(epoch_y, epoch_predictions),
        "Precision": precision_score(epoch_y, epoch_predictions, zero_division=0),
        "Recall": recall_score(epoch_y, epoch_predictions, zero_division=0),
        "F1": f1_score(epoch_y, epoch_predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(epoch_y, epoch_probabilities),
    }
    return metrics, epoch_y, epoch_predictions, epoch_probabilities


def plot_outputs(metrics_df, results):
    sns.set_theme(style="whitegrid")
    metric_columns = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
    long_metrics = metrics_df.melt(id_vars="Condition", value_vars=metric_columns,
                                   var_name="Metric", value_name="Score")
    plt.figure(figsize=(10, 6))
    ax = sns.barplot(data=long_metrics, x="Metric", y="Score", hue="Condition", palette="Set2")
    ax.set_ylim(0, 1)
    ax.set_title("DeepMLP: raw vs ICA-cleaned EEG classification")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=2, fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "raw_vs_clean_classification_metrics.png", dpi=200)
    plt.close()

    for condition, (_, y_true, y_pred, _) in results.items():
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=["Focused", "MW"], yticklabels=["Focused", "MW"])
        plt.title(f"{condition}: out-of-fold confusion matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        safe_name = condition.lower().replace("-", "_")
        plt.savefig(OUT_DIR / f"{safe_name}_confusion_matrix.png", dpi=200)
        plt.close()



def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    label_csv = Path(LABEL_CSV)
    if not label_csv.exists():
        raise FileNotFoundError(f"Set LABEL_CSV correctly. Not found: {label_csv}")
    df_main = pd.read_csv(label_csv)
    required = {"Subject", "Session", "Epoch_Index", "State"}
    missing = required - set(df_main.columns)
    if missing:
        raise ValueError(f"df_main_phase1.csv is missing columns: {sorted(missing)}")

    dataset = make_dataset(df_main)
    if dataset.empty:
        raise RuntimeError("No paired raw/clean subjects were processed. Check the three input paths.")
    results = {}
    for condition in ("Raw", "ICA-cleaned"):
        metrics, y_true, y_pred, probabilities = evaluate_condition(dataset, condition)
        results[condition] = (metrics, y_true, y_pred, probabilities)

    metrics_df = pd.DataFrame([result[0] for result in results.values()])
    metrics_df.to_csv(OUT_DIR / "raw_vs_clean_metrics.csv", index=False)
    plot_outputs(metrics_df, results)
    print("\nCompleted. Outputs written to:", OUT_DIR)
    print(metrics_df.to_string(index=False))


if __name__ == "__main__":
    main()


Subject 1: using 40 paired epochs (raw=40, clean=40)
Finished subject 1
Subject 2: using 40 paired epochs (raw=40, clean=40)
Finished subject 2
Subject 3: using 40 paired epochs (raw=40, clean=40)
Finished subject 3
Subject 5: using 40 paired epochs (raw=40, clean=40)
Finished subject 5
Subject 7: using 40 paired epochs (raw=40, clean=40)
Finished subject 7
Subject 8: using 40 paired epochs (raw=40, clean=40)
Finished subject 8
Subject 9: using 40 paired epochs (raw=40, clean=40)
Finished subject 9
Subject 10: using 40 paired epochs (raw=40, clean=40)
Finished subject 10
Subject 11: using 40 paired epochs (raw=40, clean=40)
Finished subject 11
Subject 12: using 40 paired epochs (raw=40, clean=40)
Finished subject 12
Subject 13: using 40 paired epochs (raw=40, clean=40)
Finished subject 13
Subject 14: using 40 paired epochs (raw=40, clean=40)
Finished subject 14
Subject 15: using 40 paired epochs (raw=40, clean=40)
Finished subject 15
Subject 16: using 40 paired epochs (raw=40, clean=40